# Day 20 — API Validation and Error Handling

**Project:** IPL Player Performance Analytics and Prediction

**Goal:** Test Player List and Player Detail endpoint behavior without depending on production staging tables.

The previous run showed that `python manage.py check` passes, but both tests failed because the Django test database `test_ipl_analytics` did not contain `stg_ipl_batting_player_match`. This corrected notebook uses mocked database-helper results for unit tests, so it does not require those tables in the Django test database.

**Prerequisites:** Day 18 Player APIs and Day 19 URL registration are in place. Run this notebook from the repository root using the `ipl-player-performance` environment.

In [1]:
from pathlib import Path
import os, sys, subprocess, csv
from datetime import datetime

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "django_api").is_dir():
    roots = [p for p in [REPO_ROOT, *REPO_ROOT.parents] if (p / "django_api").is_dir()]
    if roots:
        REPO_ROOT = roots[0]
DJANGO_ROOT = REPO_ROOT / "django_api"
MANAGE_PY = DJANGO_ROOT / "manage.py"
VIEWS_PATH = DJANGO_ROOT / "ipl_api" / "views.py"
APP_URLS_PATH = DJANGO_ROOT / "ipl_api" / "urls.py"
PROJECT_URLS_PATH = DJANGO_ROOT / "config" / "urls.py"
DOCS_DIR = REPO_ROOT / "docs"
REPORT_DIR = REPO_ROOT / "reports" / "day20"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
for p in (MANAGE_PY, VIEWS_PATH, APP_URLS_PATH, PROJECT_URLS_PATH, DJANGO_ROOT / "config" / "settings.py"):
    print(("FOUND " if p.exists() else "MISSING "), p)
if not all(p.exists() for p in (MANAGE_PY, VIEWS_PATH, APP_URLS_PATH, PROJECT_URLS_PATH)):
    raise FileNotFoundError("Required Django files are missing. Complete Days 18–19 first.")


Repository: D:\Git\Personal\ipl-player-performance
FOUND  D:\Git\Personal\ipl-player-performance\django_api\manage.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\views.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\config\urls.py
FOUND  D:\Git\Personal\ipl-player-performance\django_api\config\settings.py


In [2]:
# Inspect the real API code before writing tests.
views_text = VIEWS_PATH.read_text(encoding="utf-8")
app_urls_text = APP_URLS_PATH.read_text(encoding="utf-8")
project_urls_text = PROJECT_URLS_PATH.read_text(encoding="utf-8")
settings_text = (DJANGO_ROOT / "config" / "settings.py").read_text(encoding="utf-8")

print("--- ipl_api/urls.py ---")
print(app_urls_text)
print("--- config/urls.py ---")
print(project_urls_text)
print("--- ipl_api/views.py (first 240 lines) ---")
print("\n".join(views_text.splitlines()[:240]))
print("DRF appears in settings:", "rest_framework" in settings_text)
print("ipl_api appears in settings:", "ipl_api" in settings_text)


--- ipl_api/urls.py ---
from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

urlpatterns = [
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
]

--- config/urls.py ---
from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path(
        "",
        RedirectView.as_view(url="/api/dashboard/", permanent=False),
        name="home",
    ),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin.site.urls),
]

--- ipl_api/views.py (first 240 lines) ---
from django.db import connection
from django.http import Http404
from rest_framework import status
from rest_framework.response import Response
from rest_framework.views import APIView

# Table and column names detected from the MySQL staging schema.
BATTING_TABLE = 'stg_ipl_batting_player_match'

## Test strategy

The uploaded run shows `manage.py check` passes, but the two tests fail because Django creates an empty test database (`test_ipl_analytics`) that does not contain the staging tables. These API views use raw SQL, so the tests must mock the database helper rather than expect the staging tables in Django's test database.

The corrected tests use `SimpleTestCase` plus `unittest.mock.patch` to isolate endpoint behavior:
- Player List: returns a controlled paginated JSON payload.
- Player Detail: returns a controlled aggregate payload for a known player.
- Unknown player: returns HTTP 404 when both mocked aggregate queries contain no records.

This validates API behavior without creating or changing database tables. It does not replace a separate integration test against the real MySQL staging tables.

In [3]:
# Create a DB-independent test module. Back up an existing Day 20 test before replacing it.
TESTS_DIR = DJANGO_ROOT / "ipl_api" / "tests"
TESTS_DIR.mkdir(parents=True, exist_ok=True)
init_file = TESTS_DIR / "__init__.py"
if not init_file.exists():
    init_file.write_text("", encoding="utf-8")

TEST_PATH = TESTS_DIR / "test_player_api_day20.py"
if TEST_PATH.exists():
    backup = TEST_PATH.with_name("test_player_api_day20.py.backup_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    backup.write_text(TEST_PATH.read_text(encoding="utf-8"), encoding="utf-8")
    print("Backed up existing test to:", backup)

TEST_CODE = 'from django.test import SimpleTestCase\nfrom unittest.mock import patch\n\nclass PlayerApiSmokeTests(SimpleTestCase):\n    @patch("ipl_api.views.fetch_all")\n    def test_player_list_returns_paginated_json(self, mock_fetch_all):\n        mock_fetch_all.side_effect = [\n            [{"total": 2}],\n            [{"player_name": "Virat Kohli"}, {"player_name": "Rohit Sharma"}],\n        ]\n        response = self.client.get("/api/players/?page=1&page_size=2", HTTP_ACCEPT="application/json")\n        self.assertEqual(response.status_code, 200)\n        payload = response.json()\n        self.assertEqual(payload["count"], 2)\n        self.assertEqual(payload["page"], 1)\n        self.assertEqual(len(payload["results"]), 2)\n\n    @patch("ipl_api.views.fetch_all")\n    def test_player_detail_returns_aggregates(self, mock_fetch_all):\n        mock_fetch_all.side_effect = [\n            [{"batting_records": 1, "matches": 1, "runs": 42, "balls_faced": 30, "fours": 4, "sixes": 1}],\n            [{"bowling_records": 0, "matches": 0, "wickets": 0, "runs_conceded": 0, "overs": 0}],\n        ]\n        response = self.client.get("/api/players/Virat%20Kohli/", HTTP_ACCEPT="application/json")\n        self.assertEqual(response.status_code, 200)\n        payload = response.json()\n        self.assertEqual(payload["player_name"], "Virat Kohli")\n        self.assertEqual(payload["batting"]["runs"], 42)\n\n    @patch("ipl_api.views.fetch_all")\n    def test_unknown_player_returns_404(self, mock_fetch_all):\n        mock_fetch_all.side_effect = [\n            [{"batting_records": 0, "matches": 0, "runs": 0, "balls_faced": 0, "fours": 0, "sixes": 0}],\n            [{"bowling_records": 0, "matches": 0, "wickets": 0, "runs_conceded": 0, "overs": 0}],\n        ]\n        response = self.client.get("/api/players/__day20_unknown_player__/", HTTP_ACCEPT="application/json")\n        self.assertEqual(response.status_code, 404)\n'
TEST_PATH.write_text(TEST_CODE, encoding="utf-8")
print("Created DB-independent test module:", TEST_PATH)


Backed up existing test to: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\tests\test_player_api_day20.py.backup_20261023_213407
Created DB-independent test module: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\tests\test_player_api_day20.py


In [4]:
# Run Django checks and focused tests. SimpleTestCase + mocks should not require the staging tables.
env = os.environ.copy()
check = subprocess.run([sys.executable, str(MANAGE_PY), "check"], cwd=str(DJANGO_ROOT), capture_output=True, text=True, env=env)
print("=== python manage.py check ===")
print("Return code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)

tests = subprocess.run(
    [sys.executable, str(MANAGE_PY), "test", "ipl_api.tests.test_player_api_day20", "--verbosity", "2"],
    cwd=str(DJANGO_ROOT), capture_output=True, text=True, env=env
)
print("\\n=== Django API tests (mocked database helper; no staging-table dependency) ===")
print("Return code:", tests.returncode)
print(tests.stdout)
if tests.stderr:
    print(tests.stderr)


=== python manage.py check ===
Return code: 0
System check identified no issues (0 silenced).

\n=== Django API tests (mocked database helper; no staging-table dependency) ===
Return code: 0
Found 3 test(s).
Skipping setup of unused database(s): default.
System check identified no issues (0 silenced).

test_player_detail_returns_aggregates (ipl_api.tests.test_player_api_day20.PlayerApiSmokeTests.test_player_detail_returns_aggregates) ... ok
test_player_list_returns_paginated_json (ipl_api.tests.test_player_api_day20.PlayerApiSmokeTests.test_player_list_returns_paginated_json) ... ok
test_unknown_player_returns_404 (ipl_api.tests.test_player_api_day20.PlayerApiSmokeTests.test_unknown_player_returns_404) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.014s

OK



In [5]:
# Generate documentation from the observed command outputs.
def tail(text, count):
    return "\n".join((text or "").splitlines()[-count:])

status_rows = [
    ("generated_at", datetime.now().isoformat(timespec="seconds")),
    ("repository_root", str(REPO_ROOT)),
    ("django_check_return_code", str(check.returncode)),
    ("django_tests_return_code", str(tests.returncode)),
    ("test_file", str(TEST_PATH)),
]
with (REPORT_DIR / "task_status.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["item", "value"])
    writer.writerows(status_rows)

doc = f'''# Day 20 — API Validation, Error Handling, and Tests (mocked database helper)

- Generated: {datetime.now().isoformat(timespec="seconds")}
- Repository: `{REPO_ROOT}`
- Test file: `{TEST_PATH}`
- `manage.py check` return code: **{check.returncode}**
- Focused test command return code: **{tests.returncode}**

## Django check output

```text
{tail(check.stdout + chr(10) + check.stderr, 15)}
```

## Test output

```text
{tail(tests.stdout + chr(10) + tests.stderr, 35)}
```

## Interpretation

- Return code `0` means the command completed successfully; inspect the output for the number of tests run.
- If a test fails because your route differs from the expected pattern, update the test to match `ipl_api/urls.py`.
- If tests fail because of test database permissions, inspect the Django/MySQL test database configuration before modifying production data.
- This report reflects actual command output and does not assume that tests passed.
'''
(DOCS_DIR / "day20_api_validation_error_handling.md").write_text(doc, encoding="utf-8")
print("Saved:", REPORT_DIR / "task_status.csv")
print("Saved:", DOCS_DIR / "day20_api_validation_error_handling.md")


Saved: D:\Git\Personal\ipl-player-performance\reports\day20\task_status.csv
Saved: D:\Git\Personal\ipl-player-performance\docs\day20_api_validation_error_handling.md


## Day 20 completion checklist

- [ ] `python manage.py check` passes.
- [ ] Focused API tests complete successfully.
- [ ] Player List returns a controlled HTTP response without a server error.
- [ ] Unknown player requests return a controlled validation/not-found response.
- [ ] Review `docs/day20_api_validation_error_handling.md` and `reports/day20/task_status.csv`.

**Next:** Day 21 — add pagination and query filtering to the player list endpoint after confirming the existing API contract.